<a href="https://colab.research.google.com/github/parimal173/DIY_MRI_DLapproches/blob/main/Day_3_Notebook_1_3_zssr.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/drive')

Mounted at /drive


In [ ]:
!git clone https://github.com/ajaynice1996/DIY_MRI_Workshop_II_Recon

Cloning into 'DIY_MRI_Workshop_II_Recon'...
remote: Enumerating objects: 195, done.
remote: Counting objects: 100% (86/86), done.
remote: Compressing objects: 100% (84/84), done.
remote: Total 195 (delta 40), reused 7 (delta 1), pack-reused 109 (from 2)
Receiving objects: 100% (195/195), 115.23 MiB | 28.65 MiB/s, done.
Resolving deltas: 100% (53/53), done.


In [ ]:
pwd

'/content'

In [ ]:
!pip install nilearn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 63.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 75.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.8 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.3
    Uninstalling pandas-2.2.3:
      Successfully uninstalled pandas-2.2.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.0.5 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [ ]:
!pip install colorama
!pip -q install roipoly

  Preparing metadata (setup.py) ... done


In [ ]:
import sys

REPO_ROOT = "/content/DIY_MRI_Workshop_II_Recon"

sys.path.insert(0, REPO_ROOT)
sys.path.insert(0, f"{REPO_ROOT}/zssr")

In [ ]:
# T1 and T2 weighted images

import os
import nibabel as nib
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import zoom
from typing import Dict, Tuple
# from skimage.filters import threshold_otsu

class PairedMRI:
    def __init__(self, root_dir: str):
        """
        Args:
            root_dir: Path to training folder containing subject folders.
        """
        self.root_dir = root_dir
        self.subjects = sorted(os.listdir(root_dir))

    def _load_modality(self, path: str) -> nib.Nifti1Image:
        return nib.load(path)

    def _get_voxel_size(self, nii: nib.Nifti1Image) -> Tuple[float, float, float]:
        return tuple(np.round(nii.header.get_zooms(), 3))

    def _resample(self, data: np.ndarray, original_spacing: Tuple[float, float, float], target_spacing: Tuple[float, float, float]) -> np.ndarray:
        zoom_factors = [o/t for o, t in zip(original_spacing, target_spacing)]
        return zoom(data, zoom_factors, order=1)

    def _normalize(self, data: np.ndarray) -> np.ndarray:
        data = data.astype(np.float32)
        min_val, max_val = np.min(data), np.max(data)
        if max_val - min_val > 0:
            data = (data - min_val) / (max_val - min_val)
        return data

    def get_subject_data(self, subject_id: str) -> Dict[str, Dict[str, np.ndarray]]:
        """
        Returns all modalities for given subject as dict:
        {
            "HF": {"t2w": np.ndarray, "flair": np.ndarray, "t1w": np.ndarray},
            "LF": {"t2w": np.ndarray, "flair": np.ndarray, "t1w": np.ndarray}
        }
        """
        subj_path = os.path.join(self.root_dir, subject_id)

        hf_path = os.path.join(subj_path, "3T")
        lf_path = os.path.join(subj_path, "64mT")

        hf_data, lf_data = {}, {}
        for seq in ["FLAIR", "T1", "T2"]:
            hf_img = self._load_modality(os.path.join(hf_path, f"{subject_id}_{seq}.nii.gz"))
            lf_img = self._load_modality(os.path.join(lf_path, f"{subject_id}_{seq}.nii.gz"))

            hf_data[seq] = hf_img
            lf_data[seq] = lf_img

        return {"HF": hf_data, "LF": lf_data}

    def get_voxel_sizes(self, subject_id: str) -> Dict[str, Dict[str, Tuple[float]]]:
        data = self.get_subject_data(subject_id)
        voxel_sizes = {
            "HF": {seq: self._get_voxel_size(img) for seq, img in data["HF"].items()},
            "LF": {seq: self._get_voxel_size(img) for seq, img in data["LF"].items()},
        }
        return voxel_sizes

    def get_resampled_normalized(self, subject_id: str, target_spacing=(1.0, 1.0, 2.0)) -> Dict[str, Dict[str, np.ndarray]]:
        data = self.get_subject_data(subject_id)
        processed = {"HF": {}, "LF": {}}
        for field in ["HF", "LF"]:
            for seq, img in data[field].items():
                voxel_size = self._get_voxel_size(img)
                arr = img.get_fdata()
                resampled = self._resample(arr, voxel_size, target_spacing)
                normed = self._normalize(resampled)
                # voxel_size = self._get_voxel_size(normed)
                processed[field][seq] = normed
        return processed

    def describe_subject(self, subject_id: str):
      """
      Prints details of all HF and LF modalities for the given subject:
      - Image shape
      - Voxel size
      - Min/Max values
      - Mean/Standard Deviation
      """
      data = self.get_subject_data(subject_id)

      print(f"\n--- Subject: {subject_id} ---")
      for field in ["HF", "LF"]:
          print(f"\n{field} MRI Data:")
          for seq, img in data[field].items():
              voxel_size = self._get_voxel_size(img)
              arr = img.get_fdata()
              min_val, max_val = np.min(arr), np.max(arr)
              mean_val, std_val = np.mean(arr), np.std(arr)
              print(f"  {seq}:")
              print(f"    Shape       : {arr.shape}")
              print(f"    Voxel size  : {voxel_size} mm")
              print(f"    Min/Max     : ({min_val:.3f}, {max_val:.3f})")
              print(f"    Mean/Std    : ({mean_val:.3f}, {std_val:.3f})")

    def analyze_noise_distribution(self, subject_id: str, hf_seq: str, lf_seq: str, mask=None, bins=200):
        """
        Compare pixel distributions of HF, LF, and residual (HF - LF) MRI for a given subject & sequence.
        """
        data = self.get_subject_data(subject_id)

        hf_img = data["HF"][hf_seq]
        lf_img = data["LF"][lf_seq]

        # Convert to arrays
        hf = hf_img.get_fdata() if hasattr(hf_img, "get_fdata") else np.array(hf_img)
        lf = lf_img.get_fdata() if hasattr(lf_img, "get_fdata") else np.array(lf_img)

        # Residual
        residual = hf - lf

        # Auto-generate mask if not provided
        if mask is None:
            thr = threshold_otsu(hf)
            mask = hf > thr

        fg_idx = mask
        bg_idx = ~mask

        def stats(arr):
            return {
                "mean": np.mean(arr),
                "std": np.std(arr),
                "min": np.min(arr),
                "max": np.max(arr),
                "SNR": np.mean(arr[fg_idx]) / np.std(arr[bg_idx])
            }

        hf_stats = stats(hf)
        lf_stats = stats(lf)
        res_stats = stats(residual)

        print("\n--- Intensity Statistics ---")
        for name, s in zip(["HF", "LF", "Residual"], [hf_stats, lf_stats, res_stats]):
            print(f"{name}: mean={s['mean']:.3f}, std={s['std']:.3f}, "
                  f"min={s['min']:.3f}, max={s['max']:.3f}, SNR={s['SNR']:.2f}")

        # Plot histograms
        fig, axs = plt.subplots(2, 3, figsize=(15, 8))
        datasets = [hf, lf, residual]
        titles = ["HF MRI", "LF MRI", "Residual (HF - LF)"]

        for i, data_arr in enumerate(datasets):
            axs[0, i].hist(data_arr[fg_idx].flatten(), bins=bins, color='blue', alpha=0.7)
            axs[0, i].set_title(f"{titles[i]} - Foreground")
            axs[0, i].set_xlabel("Intensity")
            axs[0, i].set_ylabel("Count")

            axs[1, i].hist(data_arr[bg_idx].flatten(), bins=bins, color='red', alpha=0.7)
            axs[1, i].set_title(f"{titles[i]} - Background (Noise)")
            axs[1, i].set_xlabel("Intensity")
            axs[1, i].set_ylabel("Count")
            axs[1, i].set_yscale('log')


        plt.tight_layout()
        plt.show()

    def resample_nifti(self, img, target_spacing=(1.6, 1.6, 1.0)):

        # Load image
        # img = nib.load(in_file)
        data = img.get_fdata()
        affine = img.affine
        header = img.header.copy()

        # Original voxel spacing
        original_spacing = header.get_zooms()[:3]

        # Compute zoom factors
        zoom_factors = np.array(original_spacing) / np.array(target_spacing)

        # Resample
        resampled_data = zoom(data, zoom_factors, order=3)  # cubic interpolation

        # Update affine
        new_affine = affine.copy()
        for i in range(3):
            new_affine[i, i] = target_spacing[i] * np.sign(affine[i, i])

        # Create new header with updated zooms
        new_header = header.copy()
        new_header.set_zooms(target_spacing)

        # Create new image
        new_img = nib.Nifti1Image(resampled_data, new_affine, header=new_header)

        # import nibabel.viewers

        # # Display the resampled image using OrthoSlicer3D
        # nibabel.viewers.OrthoSlicer3D(new_img.get_fdata()).show()

        return new_img

    def get_subject_image(self, subject_id: str, seq: str, modality: str = "HF",
                        target_spacing=(1.6, 1.6, 1.0),
                        slice_index: int = None, cmap="gray", visible=True):
        """
        Retrieve an MRI volume from a subject for a given sequence,
        resample it to a target voxel spacing, update header, and optionally display a slice.

        Args:
            subject_id (str): Subject identifier.
            seq (str): Sequence key (e.g., "T1", "T2", etc.).
            modality (str): "HF" or "LF" (default: "HF").
            target_spacing (tuple): Desired voxel spacing in mm (dx, dy, dz).
            slice_index (int): Slice index to visualize if visible=True (default: center slice).
            cmap (str): Colormap for visualization (default: "gray").
            visible (bool): If True, display the chosen slice.

        Returns:
            nibabel.Nifti1Image: Resampled image with updated header
        """
        import nibabel as nib
        import matplotlib.pyplot as plt

        # Load subject data
        data = self.get_subject_data(subject_id)
        img = data[modality][seq]

        # Resample to target spacing
        resampled_img = self.resample_nifti(img, target_spacing=target_spacing)

        # Update header voxel sizes
        new_header = resampled_img.header.copy()
        new_header.set_zooms(target_spacing)
        resampled_img = nib.Nifti1Image(resampled_img.get_fdata(), resampled_img.affine, header=new_header)

        # Visualization
        if visible:
            if slice_index is None:
                slice_index = resampled_img.shape[2] // 2
            plt.imshow(resampled_img.get_fdata()[:, :, slice_index], cmap=cmap)
            plt.title(f"{modality}-{seq} slice {slice_index}")
            plt.axis("off")
            plt.show()

        return resampled_img


    def compare_hf_lf_alignment(self, subject_id: str, hf_seq: str, lf_seq: str, slice_index: int = None, cmap_hf="gray", cmap_lf="hot", alpha=0.5):

        """
        Compare HF and LF MRI alignment by showing HF, LF, and Overlay slices side-by-side.

        Args:
            subject_id (str): Subject identifier.
            hf_seq (str): HF sequence key (e.g., "T1", "T2").
            lf_seq (str): LF sequence key (same sequence type as HF, e.g., "T1", "T2").
            slice_index (int): Slice index to display (default: middle slice).
            cmap_hf (str): Colormap for HF image (default: "gray").
            cmap_lf (str): Colormap for LF overlay (default: "hot").
            alpha (float): Transparency for LF overlay (0=transparent, 1=opaque).

        Returns:
            tuple: (hf_volume, lf_volume) as NumPy arrays.
        """
        # Load subject data
        data = self.get_subject_data(subject_id)
        hf_img = data["HF"][hf_seq]
        lf_img = data["LF"][lf_seq]

        # Convert to numpy
        hf_vol = hf_img.get_fdata() if hasattr(hf_img, "get_fdata") else np.array(hf_img)
        lf_vol = lf_img.get_fdata() if hasattr(lf_img, "get_fdata") else np.array(lf_img)

        # Pick slice (default = middle slice)
        if slice_index is None:
            slice_index = hf_vol.shape[2] // 2

        hf_slice = hf_vol[:, :, slice_index]
        lf_slice = lf_vol[:, :, slice_index]

        # Plot side-by-side
        fig, axs = plt.subplots(1, 3, figsize=(15, 5))

        # HF
        axs[0].imshow(hf_slice.T, cmap=cmap_hf, origin="lower")
        axs[0].set_title(f"HF - {hf_seq} (slice {slice_index})")
        axs[0].axis("off")

        # LF
        axs[1].imshow(lf_slice.T, cmap=cmap_hf, origin="lower")
        axs[1].set_title(f"LF - {lf_seq} (slice {slice_index})")
        axs[1].axis("off")

        # Overlay
        axs[2].imshow(hf_slice.T, cmap=cmap_hf, origin="lower")
        axs[2].imshow(lf_slice.T, cmap=cmap_lf, origin="lower", alpha=alpha)
        axs[2].set_title("Overlay")
        axs[2].axis("off")

        plt.tight_layout()
        plt.show()

        return hf_vol, lf_vol


    def make_train_val_split(
    self,
    seq: str = "T2",   # "T1", "T2", "FLAIR", "all", or comma-separated like "T1,T2"
    train_size: int = 5,
    val_size: int = 5,
    target_spacing=(1.0, 1.0, 2.0),
    random_state: int = 42,
    mode: str = "multi"  # "multi" = stack channels, "append" = treat as separate samples
  ):
        """
        Creates train/validation splits for paired LF/HF MRI.

        Args:
            seq (str):
                - "T1", "T2", "FLAIR" -> single sequence
                - "all" -> all three [T1, T2, FLAIR]
                - "T1,T2", "T1,FLAIR", "T2,FLAIR" -> custom multi-sequence
            train_size (int): number of subjects for training
            val_size (int): number of subjects for validation
            target_spacing (tuple): voxel spacing to resample both HF and LF
            random_state (int): reproducibility
            mode (str):
                - "multi": stack sequences into channels (N, C, H, W, D)
                - "append": treat each sequence as independent (N*C, H, W, D)

        Returns:
            (x_train, y_train), (x_val, y_val)
        """
        assert train_size + val_size <= len(self.subjects), "Not enough subjects!"

        # Normalize seq argument
        if seq.lower() == "all":
            seqs = ["T1", "T2", "FLAIR"]
        else:
            seqs = [s.strip().upper() for s in seq.split(",")]

        print(f"Creating train/val split for {seqs} MRI in {mode} mode...")
        np.random.seed(random_state)
        indices = np.random.permutation(len(self.subjects))
        train_idx = indices[:train_size]
        val_idx = indices[train_size:train_size + val_size]

        def collect(indices):
            x, y = [], []
            for i in indices:
                subj_id = self.subjects[i]
                data = self.get_resampled_normalized(subj_id, target_spacing=target_spacing)

                if len(seqs) == 1:
                    # Single sequence
                    x.append(data["LF"][seqs[0]])
                    y.append(data["HF"][seqs[0]])
                else:
                    if mode == "multi":
                        # Stack as channels → (C, H, W, D)
                        lf_stack = np.stack([data["LF"][s] for s in seqs], axis=0)
                        hf_stack = np.stack([data["HF"][s] for s in seqs], axis=0)
                        x.append(lf_stack)
                        y.append(hf_stack)
                    elif mode == "append":
                        # Append as separate samples → increases N
                        for s in seqs:
                            x.append(data["LF"][s])
                            y.append(data["HF"][s])
                    else:
                        raise ValueError("mode must be 'multi' or 'append'")

            return np.array(x), np.array(y)

        x_train, y_train = collect(train_idx)
        x_val, y_val = collect(val_idx)

        return (x_train, y_train), (x_val, y_val)


    def get_self_supervised_data(
        self,
        seq: str = "all",
        train_size: int = 5,
        val_size: int = 5,
        target_spacing=(1.0, 1.0, 2.0),
        mode: str = "multi"
    ):
        """
        Returns only LF scans for self-supervised or zero-shot setups.
        Supports single, multi, or all sequences.
        mode:
            - "multi": stacked channels (N, C, H, W, D)
            - "append": independent samples (N*C, H, W, D)
        """
        (x_train, _), (x_val, _) = self.make_train_val_split(
            seq, train_size, val_size, target_spacing=target_spacing, mode=mode
        )
        return x_train, x_val

    def display_pair(self, subject_id: str, seq: str, slice_index: int = None, save_path: str = None):
        data = self.get_resampled_normalized(subject_id)
        hf_img = data["HF"][seq]
        lf_img = data["LF"][seq]

        if seq == "T1":
            s_seq = "T1w"
        elif seq == "T2":
            s_seq = "T2w"
        elif seq == "FLAIR":
            s_seq = "FLAIR"
        else:
            s_seq = seq

        if slice_index is None:
            slice_index = hf_img.shape[2] // 2

        # Flip the images horizontally (left-right)
        hf_img_flipped = np.rot90(hf_img[:, :, slice_index])
        lf_img_flipped = np.rot90(lf_img[:, :, slice_index])

        fig, axes = plt.subplots(1, 2, figsize=(8, 4))
        axes[0].imshow(lf_img_flipped, cmap="gray")
        axes[0].set_title(f"LF 0.064 T - {s_seq}")
        axes[0].axis("off")

        axes[1].imshow(hf_img_flipped, cmap="gray")
        axes[1].set_title(f"HF 3T - {s_seq}")
        axes[1].axis("off")

        plt.tight_layout()
        plt.savefig(os.path.join(save_path, f"{subject_id}_{seq}_slice{slice_index}.png"))
        plt.show()

        # if save_path:
        #     # Create directory if it doesn't exist
        #     os.makedirs(save_path, exist_ok=True)
        #     plt.savefig(os.path.join(save_path, f"{subject_id}_{seq}_slice{slice_index}.png"))
        #     print(f"Saved images to {save_path}")

    def show_hf_lf_difference(self, subject_id: str, hf_seq: str, lf_seq: str, slice_index: int = None, cmap="bwr"):

        """
        Show the difference image (HF - LF) for a given subject and sequence.

        Args:
            subject_id (str): Subject identifier.
            hf_seq (str): HF sequence key (e.g., "T1", "T2").
            lf_seq (str): LF sequence key (same sequence type as HF).
            slice_index (int): Slice index to display (default: middle slice).
            cmap (str): Colormap for difference visualization (default: "bwr" -> blue=negative, red=positive).

        Returns:
            np.ndarray: Difference volume (HF - LF).
        """
        # Load subject data
        data = self.get_subject_data(subject_id)
        hf_img = data["HF"][hf_seq]
        lf_img = data["LF"][lf_seq]

        # Convert to numpy
        hf_vol = hf_img.get_fdata() if hasattr(hf_img, "get_fdata") else np.array(hf_img)
        lf_vol = lf_img.get_fdata() if hasattr(lf_img, "get_fdata") else np.array(lf_img)

        # Compute difference
        diff_vol = hf_vol - lf_vol

        # Pick slice
        if slice_index is None:
            slice_index = hf_vol.shape[2] // 2

        diff_slice = diff_vol[:, :, slice_index]

        # Plot
        plt.figure(figsize=(6, 6))
        plt.imshow(diff_slice.T, cmap="Reds", origin="lower")
        plt.colorbar(label="HF - LF Intensity")
        plt.title(f"Difference Image (slice {slice_index})")
        plt.axis("off")
        plt.show()

        return diff_vol

if __name__ == "__main__":

        # Improvement in difference image discard outside things

        dataset = PairedMRI("/content/DIY_MRI_Workshop_II_Recon/data/Training_data")

        # # List subjects
        # print(dataset.subjects)
        # # Get voxel sizes
        # voxel_info = dataset.get_voxel_sizes(dataset.subjects[0])
        # print(voxel_info)
        # dataset.describe_subject(dataset.subjects[0])
        # dataset.get_resampled_normalized((dataset.subjects[0]))
        # dataset.display_pair(dataset.subjects[1], "T1", save_path= "Data/")
        # # dataset.analyze_noise_distribution("POCEMR003", hf_seq="T1", lf_seq="T1")

        # dataset.get_subject_image(dataset.subjects[0], "T1", 'LF')
        # # dataset.compare_hf_lf_alignment(dataset.subjects[4], "T1", "T1")
        # # dataset.show_hf_lf_difference(dataset.subjects[4], "T1", "T1")
        # # # Multi-channel (default) → shape (N, 2, H, W, D)
        # # (x_train, y_train), (x_val, y_val) = dataset.make_train_val_split("T1", train_size=1, val_size=1, mode="multi")
        # # print(x_train.shape, y_train.shape)
        # # print(x_val.shape, y_val.shape)

        # # subject = dataset.get_subject_image(dataset.subjects[0], "T1", 'LF', visible=True)
        # # subject = dataset.get_subject_image(dataset.subjects[0], "T1", 'HF', visible=True)

        # # print(f"Shape: {subject.shape}")
        # # print(f"Data type: {subject.dtype}")
        # # print(f"Min: {np.min(subject)}, Max: {np.max(subject)}")
        # # print(f"Mean: {np.mean(subject):.3f}, Std: {np.std(subject):.3f}")

In [ ]:
import nibabel as nib
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import zoom
from typing import Dict, Tuple

# SRR_SS_Mon
from SRR_SS_Mon.data_read import PairedMRI

# ZSSR
from zssr.ZSSR_master import ZSSR
from zssr.ZSSR_master import configs, configs_2

# Utilities
from zssr.utils import compute_aes

In [ ]:
from nilearn import plotting
from nibabel.viewers import OrthoSlicer3D
import tensorflow as tf
# import pydicom # Commented out as it was previously for potential import errors
import numpy as np
import matplotlib
# matplotlib.use('TkAgg')  # or 'Qt5Agg' depending on what's installed
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim
# from pydicom.filereader import dcmread # Commented out as it was previously for potential import errors
from tensorflow.keras import backend as K
import scipy.io as sio

# Clear the current TensorFlow/Keras session
K.clear_session()

print(tf.config.list_physical_devices('GPU'))  # Check if GPU is visible
print(tf.config.list_physical_devices('CPU'))  # Check if CPU is visible

if tf.config.list_physical_devices('GPU'):
    print("CUDNN detected!")
else:
    print("Display the data using OrthoSlicer3D")
    # Removed img.dataobj as img is not defined here yet
    # OrthoSlicer3D(img.dataobj).show()
    # plotting.plot_anat(img, title="3D TSC Image")
    plt.show()

[]
[PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU')]
Display the data using OrthoSlicer3D


In [ ]:
import warnings
# Suppress DeprecationWarnings from jupyter_client
warnings.filterwarnings('ignore', category=DeprecationWarning, module='jupyter_client')
print("DeprecationWarnings from jupyter_client are now suppressed.")

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

DeprecationWarnings from jupyter_client are now suppressed.


In [ ]:
# Note: Undo CUDNN detected!")

from do_zssr_collage import *
from nifti_write import make_nifti
# from LF_simulation_functions import read_nifti
from colorama import Fore, Back, Style
import itertools
from scipy.ndimage import sobel

In [ ]:
viewing = False
ds_to_process = 4
target_resolution_fact = [1, 1, 2]
scale_factor = target_resolution_fact[2]  # Z-axis scaling factor
snr_component = False

max_iters = 10
min_iters = 5
# Define parameter options
widths = [32] # width of the filters in the conv layers
depths = [4] # depth of the network
crop_sizes = [32] # size of the patches to crop from the image
noise_stds = [0.0] # standard deviation of the noise to add to the image

In [ ]:
# Load dataset
training_path = "/content/DIY_MRI_Workshop_II_Recon/data/Training_data"
dataset = PairedMRI(training_path)
kernel_path = '/Users/sairamgeethanath/Documents/Contributions/Tools/Projects/R21/lf-brain-tracking/src/ZSSR_master/kernel_example/BSD100_100_lr_rand_ker_c_X2_0.mat'

kernel_files = ['%s_%d.mat' % (kernel_path[:-4], ind) for ind in range(len([1, 2]))]
# List subjects
print(dataset.subjects)

for i, subject_id in enumerate(dataset.subjects[0:1]):  # take first 5 subjects
    print(Fore.CYAN + f"\n=== Processing Subject {i+1}: {subject_id} ===" + Style.RESET_ALL)

    # Get LF & HF images
    subject_LF_Monash = dataset.get_subject_image(subject_id, "T1", 'LF', visible=False, target_spacing=
                                           [1, 1, 1])
    subject_LF_ZSSR = dataset.get_subject_image(subject_id, "T1", 'LF', visible=False,
                                           target_spacing=target_resolution_fact)
    subject_HF = dataset.get_subject_image(subject_id, "T1", 'HF', visible=False,target_spacing=
                                           [1, 1, 1])

    # Convert to uint8 NIfTI
    subject_LF_Monash = nib.Nifti1Image(subject_LF_Monash.get_fdata().astype(np.uint8),
                                 subject_LF_Monash.affine, subject_LF_Monash.header)
    subject_LF_ZSSR = nib.Nifti1Image(subject_LF_ZSSR.get_fdata().astype(np.uint8),
                                 subject_LF_ZSSR.affine, subject_LF_ZSSR.header)
    subject_HF = nib.Nifti1Image(subject_HF.get_fdata().astype(np.uint8),
                                 subject_HF.affine, subject_HF.header)

    img_data = subject_LF_ZSSR.get_fdata()

    if viewing:
        print("Displaying LF ZSSR image data using OrthoSlicer3D")
        OrthoSlicer3D(img_data).show()

    print("Shape of img_data:", img_data.shape)
    # Generate unique NIfTI filename per subject

    nifti_file = f"Data/{subject_id}_T1.nii.gz"

    hdr = subject_LF_ZSSR.header
    pixdim = hdr['pixdim']

    # Print info
    print(Fore.GREEN + 'PROCESSING NIFTI FILE METADATA' + Style.RESET_ALL)
    print("Dimensions:", hdr.get_data_shape())
    print("Voxel Sizes:", hdr.get_zooms())
    print("Data Type:", hdr.get_data_dtype())
    print("Intent:", hdr.get_intent())

    # Get the HF data for comparison
    subject_HF_data = subject_HF.get_fdata()
    subject_HF_data = subject_HF_data / np.max(subject_HF_data)
    subject_HF_data = (subject_HF_data * 4095).astype(np.uint16)

    subject_LF_Monash_data = subject_LF_Monash.get_fdata()
    subject_LF_Monash_data = subject_LF_Monash_data / np.max(subject_LF_Monash_data)
    subject_LF_Monash_data = (subject_LF_Monash_data * 4095).astype(np.uint16)

    # start a clock so that we can compute the time taken for all parameter combinations
    import time
    start_time = time.time()
    # Create all combinations
    param_combinations = list(itertools.product(widths, depths, crop_sizes, noise_stds))
    # print(param_combinations)
    for idx, (width, depth, crop_size, noise_std) in enumerate(param_combinations):
        # Print current combination
        print(Fore.YELLOW + f"Running ZSSR with width={width}, depth={depth}, crop_size={crop_size}, noise_std={noise_std}, idx = {idx}" + Style.RESET_ALL)

        # change of recon.config
        recon_config = configs.Config(width=width, depth=depth, crop_size=crop_size, noise_std=noise_std)
        # recon_config.scale_factors = [[np.sqrt(target_resolution_fact[0]), 1]]
        recon_config.scale_factors = [[(target_resolution_fact[0]), 1]]
        recon_config.max_iters = max_iters
        recon_config.min_iters = min_iters
        recon_config.width = width
        recon_config.depth = depth
        recon_config.noise_std = noise_std
        recon_config.crop_size = crop_size
        num_rows = 16
        num_cols = 14
        print('Interpolation method:', recon_config.upscale_method)
        # Run ZSSR

        print('Passing through ZSSR ..........')

        im_lf_sim_zssr = do_ZSSR_steps(
            img=img_data, recon_conf=recon_config, num_cols=num_cols, num_rows=num_rows,
            fname_zssr=nifti_file, fspec='', scale_fact=scale_factor, dims = 1, ground_truth=None, kernel=None)

        # Compute PSNR/SSIM/AES between im_lf_sim_zssr and subject_HF
        # (Assuming subject_HF is already loaded as a NIfTI image)

        # Ensure all images are in the same dynamic range 0 - 1
        im_lf_sim_zssr_yz = im_lf_sim_zssr / np.max(im_lf_sim_zssr)

        print(Fore.GREEN + "Shape of im_lf_sim_zssr_yz:" + str(im_lf_sim_zssr_yz.shape) + Style.RESET_ALL)

        # Now let us switch the two axes to also perform ZSSR in the other plane
        img_data_xz = np.swapaxes(img_data, 0, 1)
        print(Fore.GREEN + "Shape of img_data_xz:" + str(img_data_xz.shape) + Style.RESET_ALL)

        # Run ZSSR on the swapped axes
        im_lf_sim_zssr_xz = do_ZSSR_steps(
            img=img_data_xz, recon_conf=recon_config, num_cols=num_cols, num_rows=num_rows,
            fname_zssr=nifti_file, fspec='', scale_fact=scale_factor, dims=1, ground_truth=None, kernel=None)

        # Swap axes back to original orientation
        im_lf_sim_zssr_xz = np.swapaxes(im_lf_sim_zssr_xz, 0, 1)

        # Combine the two ZSSR results
        # Combine the two ZSSR results by selecting, for each voxel, the value from the volume (yz or xz)
        # that has the higher local gradient magnitude (i.e., sharper neighborhood)

        # Compute gradient magnitude for both volumes
        grad_yz = np.sqrt(
            sobel(im_lf_sim_zssr_yz, axis=0, mode='reflect')**2 +
            sobel(im_lf_sim_zssr_yz, axis=1, mode='reflect')**2 +
            sobel(im_lf_sim_zssr_yz, axis=2, mode='reflect')**2
        )
        grad_xz = np.sqrt(
            sobel(im_lf_sim_zssr_xz, axis=0, mode='reflect')**2 +
            sobel(im_lf_sim_zssr_xz, axis=1, mode='reflect')**2 +
            sobel(im_lf_sim_zssr_xz, axis=2, mode='reflect')**2
        )

        # For each voxel, pick the value from the sharper (higher gradient) volume
        mask = grad_yz >= grad_xz
        im_lf_sim_zssr_combined = np.where(mask, im_lf_sim_zssr_yz, im_lf_sim_zssr_xz)

        # Make sure all comparisons are between 0 to 4095 to match 12 bit DICOM range
        im_lf_sim_zssr_combined = (im_lf_sim_zssr_combined * 4095).astype(np.uint16)
        im_lf_sim_zssr = im_lf_sim_zssr_combined

        # Compute PSNR
        psnr_value_monash = psnr(subject_LF_Monash_data, subject_HF_data)
        psnr_value_zssr = psnr(im_lf_sim_zssr, subject_HF_data)

        # Compute SSIM
        ssim_value_monash = ssim(subject_LF_Monash_data, subject_HF_data, data_range=4095)
        ssim_value_zssr = ssim(im_lf_sim_zssr, subject_HF_data, data_range=4095)

        # Compute AES
        aes_value_HF = compute_aes(subject_HF_data)
        aes_value_monash = compute_aes(subject_LF_Monash_data)
        aes_value_zssr = compute_aes(im_lf_sim_zssr)

        # Print PSNR, SSIM, and AES values in a table format
        print(Fore.GREEN + f"{'Method':<15}{'PSNR':<15}{'SSIM':<15}{'AES':<15}" + Style.RESET_ALL)
        print(Fore.GREEN + f"{'Monash LF':<15}{psnr_value_monash:<15.4f}{ssim_value_monash:<15.4f}{aes_value_monash:<15.4f}" + Style.RESET_ALL)
        print(Fore.GREEN + f"{'ZSSR':<15}{psnr_value_zssr:<15.4f}{ssim_value_zssr:<15.4f}{aes_value_zssr:<15.4f}" + Style.RESET_ALL)
        print(Fore.GREEN + f"{'HF (Ground Truth)':<15}{'N/A':<15}{'N/A':<15}{aes_value_HF:<15.4f}" + Style.RESET_ALL)
        # # Save output with subject-specific name and config values
        zssr_fname = (
            f"./Data/Results_ss/{subject_id}_T1_zssr_w{width}_d{depth}_c{crop_size}_n{noise_std}_test1{snr_component}.nii.gz"
        )

        # make_nifti(im_lf_sim_zssr, fname=zssr_fname, mask=False,
        #            res=[pixdim[1], pixdim[2], pixdim[3]], dim_info=[0, 1, 2])

        # print(Fore.YELLOW + f"Saved ZSSR output -> {zssr_fname}" + Style.RESET_ALL)
        viewing = False
        if viewing:
            # Display a panel of the mid coronal slice for HF, Monash LF, LF input, and LF ZSSR
            mid_slice = subject_HF_data.shape[1] // 2

            fig, axes = plt.subplots(1, 4, figsize=(16, 4))
            axes[0].imshow(np.rot90(subject_HF_data[:, mid_slice, :], k=1), cmap='gray')
            axes[0].set_title('High Field (HF)')
            axes[0].axis('off')

            axes[1].imshow(np.rot90(img_data[:, mid_slice, :], k=1), cmap='gray')
            axes[1].set_title('LF Input')
            axes[1].axis('off')

            axes[2].imshow(np.rot90(subject_LF_Monash_data[:, mid_slice, :], k=1), cmap='gray')
            axes[2].set_title('Monash LF')
            axes[2].axis('off')

            axes[3].imshow(np.rot90(im_lf_sim_zssr[:, mid_slice, :], k=1), cmap='gray')
            axes[3].set_title('LF ZSSR')
            axes[3].axis('off')

            plt.tight_layout()
            plt.show()

        if viewing:
            OrthoSlicer3D(im_lf_sim_zssr).show()
            plt.show()
    end_time = time.time()
    total_time = end_time - start_time
    print(Fore.CYAN + f"Total time for all parameter combinations: {total_time:.2f} seconds" + Style.RESET_ALL)

['POCEMR001', 'POCEMR003', 'POCEMR004', 'Phantom_2.nii.gz', 'phantom_1.nii.gz']

=== Processing Subject 1: POCEMR001 ===
Shape of img_data: (224, 224, 80)
PROCESSING NIFTI FILE METADATA
Dimensions: (224, 224, 80)
Voxel Sizes: (np.float32(1.0), np.float32(1.0), np.float32(2.0))
Data Type: uint8
Intent: ('none', (), '')
Running ZSSR with width=32, depth=4, crop_size=32, noise_std=0.0, idx = 0
Interpolation method: lanczos3
Passing through ZSSR ..........
(3584, 1120)
Num GPUs Available:  0
** Start training for sf= [1, np.float64(1.4142135623730951)]  **
sf: [1.         1.41421356] , iteration:  0 , loss:  1.0779935
iteration:  0 reconstruct mse: 0.00023876632154784058 , true mse: None
